# Ноутбук создан для 4 пункта чеклиста (создание и исследование классических моделей)

Препроцессинг и CV-харнесс вынесены в preprocessing.py и validation.py (общие для этого и simple_baseline_and_experimentations ноутбуков), чтобы не дублировать код. Загрузим данные и применим зафиксированный препроцессинг

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from preprocessing import preprocess_data_advanced
from validation import cross_validate_model, evaluate_model, train_kfold_and_predict

In [ ]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")
test_passenger_ids = test_data['PassengerId']

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

X_train = train_data.drop(['Survived'], axis=1)
y_train = train_data['Survived']
X_test = test_data

X_train.shape, X_test.shape

Заведём словарь для результатов всех моделей, пригодится для финального сравнения (пункт чеклиста "сложить все результаты и сравнить в конце"). `evaluate_model` из validation.py прогоняет модель через CV, печатает mean/std и сохраняет scores в results по ключу-названию

In [ ]:
results = {}